In [ ]:
############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')
time.sleep(10)

############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

In [1]:
import time
from time import sleep, monotonic
import datetime
import numpy as np
import matplotlib.pyplot as plt
import sys
import pyvisa
import qcodes as qc
from qcodes.dataset import Measurement
from qcodes.dataset import do0d
from qcodes.dataset.experiment_container import new_experiment, load_experiment_by_name
from qcodes.dataset.plotting import plot_by_id
from qcodes.dataset.data_set import load_by_id, load_by_counter
from qcodes import initialise_or_create_database_at, new_data_set, new_experiment
from qcodes.station import Station
from ipywidgets import interact, IntSlider, fixed

n_cooldown = 13
date = '2026-06-15'


initialise_or_create_database_at(f"./{date}_SNSPD{n_cooldown}.db")
import snspd
params = snspd.snspd(f'snspd{n_cooldown}.yaml')

# Set up experiment
exp_name = f'SNSPD{n_cooldown}_{date}'
sample_name = '00'

try:
    exp = qc.load_experiment_by_name(exp_name, sample=sample_name)
    print('Experiment loaded. Last ID no:', exp.last_counter)
except ValueError:
    exp = new_experiment(exp_name, sample_name)
    print('Started new experiment')

Logging hadn't been started.
Activating auto-logging. Current session state plus future input saved.
Filename       : C:\Users\QNL\.qcodes\logs\command_history.log
Mode           : append
Output logging : True
Raw input log  : False
Timestamping   : True
State          : active
Qcodes Logfile : C:\Users\QNL\.qcodes\logs\260616-40708-qcodes.log
Experiment loaded. Last ID no: 1


In [89]:
import importlib
importlib.reload(snspd)
params = snspd.snspd(f'snspd{n_cooldown}.yaml')

In [64]:
station = Station(config_file="friesland.yaml")
dmm = station.load_instrument("dmm", revive_instance=True)
yoko = station.load_instrument("yoko", revive_instance=True)
laser = station.load_instrument("laser", revive_instance=True)
MS = station.load_instrument("osc", revive_instance=True)
# pm100d = station.load_instrument("pm100d", revive_instance=True) 
pms120 = station.load_instrument("pms120", revive_instance=True)
tc = station.load_instrument("fridge", revive_instance=True)
p_att = station.load_instrument("dmm_keithley", revive_instance=True) # excluding from snapshot because none of the parameters work anyway

In [18]:
import inspect
print(inspect.signature(params.critical_current))

(device, dmm=None, yoko=None, tc=None, station=None)


Bias arrangement for critical current sweep ID 1: 

|BNC | Yoko | DMM |
|---|---|---|
|9 | SIG | N/A |  
|10 | GND | N/A |
|11 | N/A | SIG |
|12 | N/A | GND |  


In [19]:
params.critical_current(device=params.device_line_2, yoko=yoko, dmm=dmm, tc=tc)

update station
Starting experimental run with id: 1. 
1
Starting current 0.0
Starting current -2.5e-07
Starting current -5e-07
Starting current -7.5e-07
Starting current -1e-06
Starting current -1.25e-06
Starting current -1.5e-06
Starting current -1.75e-06
Starting current -2e-06
Starting current -2.25e-06
Starting current -2.5e-06
Starting current -2.75e-06
Starting current -3e-06
Starting current -3.25e-06
Starting current -3.5e-06
Starting current -3.75e-06
Starting current -4e-06
Starting current -4.25e-06
Starting current -4.5e-06
Starting current -4.75e-06
Starting current -5e-06
Starting current -5.25e-06
Starting current -5.5e-06
Starting current -5.75e-06
Starting current -6e-06
Starting current -6.25e-06
Starting current -6.5e-06
Starting current -6.75e-06
Starting current -7e-06
Starting current -7.25e-06
Starting current -7.5e-06
Starting current -7.75e-06
Starting current -8e-06
Starting current -8.25e-06
Starting current -8.5e-06
Starting current -8.75e-06
Starting curren

In [20]:
print(inspect.signature(params.ramp_yoko_current))

(yoko, target, step)


In [22]:
params.ramp_yoko_current(yoko, target=0, step=0.5e-6)
yoko.current(0)

Ramping to 0


In [30]:
load_by_id(1).metadata['device']

'R7C6'

In [9]:
params.args(params.critical_current)

(device, dmm=None, yoko=None, tc=None, unlatch=True, station=None)


In [14]:
params.critical_current(params.device_line_2, dmm, yoko, tc)

update station
Ramping to 0
Unlatch wait time 60
Ramping to 0.0
Starting experimental run with id: 2. 
2
Starting current 0.0
Starting current -2.5e-07
Starting current -5e-07
Starting current -7.5e-07
Starting current -1e-06
Starting current -1.25e-06
Starting current -1.5e-06
Starting current -1.75e-06
Starting current -2e-06
Starting current -2.25e-06
Starting current -2.5e-06
Starting current -2.75e-06
Starting current -3e-06
Starting current -3.25e-06
Starting current -3.5e-06
Starting current -3.75e-06
Starting current -4e-06
Starting current -4.25e-06
Starting current -4.5e-06
Starting current -4.75e-06
Starting current -5e-06
Starting current -5.25e-06
Starting current -5.5e-06
Starting current -5.75e-06
Starting current -6e-06
Starting current -6.25e-06
Starting current -6.5e-06
Starting current -6.75e-06
Starting current -7e-06
Starting current -7.25e-06
Starting current -7.5e-06
Starting current -7.75e-06
Starting current -8e-06
Starting current -8.25e-06
Starting current -8

In [18]:
params.args(params.trace_vs_current)

(device, MS, dmm, yoko, trigger, v_scale, wait=120, currents=None, unlatch=True, station=None)


Sigh...the data set is missing current 13uA. Adding extra trigger value on the end. CHecked the SNSPD12-Line2-Count_Calibration.ipynb notebook and it is because the ID range  was set to range1 = np.arange(121, 173) which excludes ID 173 but it is ID 173 that contains the 13uA value. Regenerating data set to copy to config file. 

Note this run saved a dummy variable for p_att because it was not working at the time and was not necessary. 

In [31]:
trigger = [0.012, 0.012, 0.012, 0.012, 0.012, 0.012, 0.012, 0.012, 0.012,
       0.012, 0.012, 0.048, 0.048, 0.048, 0.048, 0.048, 0.048, 0.048,
       0.048, 0.048, 0.048, 0.102, 0.102, 0.102, 0.102, 0.102, 0.102,
       0.102, 0.102, 0.102, 0.102, 0.102, 0.102, 0.102, 0.102, 0.102,
       0.102, 0.102, 0.102, 0.102, 0.102, 0.102, 0.102, 0.102, 0.102,
       0.102, 0.102, 0.102, 0.102, 0.102, 0.102, 0.102,0.102]
v_scale = 150e-3 # from SNSPD12.dB calibration measurements 
params.trace_vs_current(params.device_line_2, MS, dmm, yoko, trigger, v_scale)

Oscilloscope set for trace capture
list is trigger
single vscale value
update station
Ramping to 0
Unlatch wait time 60
Starting experimental run with id: 4. 
4


  0%|                                                                                           | 0/53 [00:00<?, ?it/s]

Check: trigger 0.012 v_scale 0.15
Oscilloscope set for trace capture
update station
Acquisition took 2.02 seconds
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 256V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;955.000E-12;188;"V";40.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1


  2%|█▌                                                                                 | 1/53 [00:19<16:39, 19.23s/it]

Check: trigger 0.012 v_scale 0.15
Oscilloscope set for trace capture
update station
Acquisition took 2.03 seconds
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 256V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;955.000E-12;188;"V";40.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1


...

Check: trigger 0.102 v_scale 0.15
Oscilloscope set for trace capture
update station
Acquisition took 2.01 seconds
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 256V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;955.000E-12;188;"V";40.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1


 98%|████████████████████████████████████████████████████████████████████████████████▍ | 52/53 [13:20<00:17, 17.54s/it]

Check: trigger 0.102 v_scale 0.15
Oscilloscope set for trace capture
update station
Acquisition took 2.03 seconds
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 256V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;955.000E-12;188;"V";40.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1


100%|██████████████████████████████████████████████████████████████████████████████████| 53/53 [13:39<00:00, 15.47s/it]


Ramping to 0
Unlatch wait time 60


Need to write function to generate the interactive plot with this. 

Bug in code meant that the triggers for all results were 0.012, and also vertical scale was saved as 1V. Can look at resolution of results to see what it actually was (25*smallest step). 

Dummy run to test the changes

In [36]:
params.args(params.trace_vs_current)

(device, MS, dmm, yoko, trigger, v_scale, wait=120, currents=None, unlatch=True, station=None)


In [37]:
trigger = [0.048, 0.102]
v_scale = 150e-3 # from SNSPD12.dB calibration measurements 
currents = [-10e-6, -11e-6]
params.trace_vs_current(params.device_line_2, MS, dmm, yoko, trigger, v_scale, currents=currents, unlatch=False)

Oscilloscope set for trace capture
list is trigger
single vscale value
update station
Ramping to -1e-05
Starting experimental run with id: 5. 
5


  0%|                                                                                            | 0/2 [00:00<?, ?it/s]

Oscilloscope set for trace capture
update station
Acquisition took 2.02 seconds
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 38.4V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;640.00E-12;188;"V";6.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1


 50%|██████████████████████████████████████████                                          | 1/2 [00:09<00:09,  9.25s/it]

Oscilloscope set for trace capture
update station
Acquisition took 2.01 seconds
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 38.4V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;975.000E-12;188;"V";6.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1


100%|████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:18<00:00,  9.24s/it]


Printed in analysis file, result is correct. 

In [38]:
params.args(params.MSO5_counts_vs_current)

(device, n_captures=10, interval=1, osc=None, dmm=None, yoko=None, pmeter90=None, currents=None, station=None)


In [70]:
params.optics_set_standard(laser, pmeter90=pms120, p_att=p_att, wavelength=1550e-9, laser_power=7, v_attenuator=5)

2026-06-16 11:40:43,364 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Power: 7.0
Frequency coarse: 193.4144THz
Wavelength (calculated) is 1550.000713493928nm
Powermeter wavelength is 1.55e-06


In [99]:
params.initialize_station()

In [75]:
############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: False


In [ ]:
params.optics_set_standard(laser, pmeter90=pms120, p_att=p_att, wavelength=1550e-9, laser_power=7, v_attenuator=5)
time.sleep(5)

In [87]:

############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')
time.sleep(10)

params.MSO5_counts_vs_current(device=params.device_line_2, n_captures=10, interval=1, osc=MS, dmm=dmm, yoko=yoko, pmeter90=pms120, unlatch=False)

############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: True
Set standard oscilloscope parameters for counts
update station
Starting experimental run with id: 10. 
10


  0%|                                                                                           | 0/53 [00:00<?, ?it/s]

0.012
This acquisition will take 10s
11 53


2026-06-16 11:53:26,664 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()

  2%|█▌                                                                                 | 1/53 [00:22<19:14, 22.20s/it]

0.012
This acquisition will take 10s
11 53


...

0.111
This acquisition will take 10s
12 7


2026-06-16 12:07:30,236 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()

 74%|████████████████████████████████████████████████████████████▎                     | 39/53 [14:25<05:10, 22.19s/it]

0.012


 74%|████████████████████████████████████████████████████████████▎                     | 39/53 [14:32<05:13, 22.38s/it]
2026-06-16 12:07:37,333 ¦ qcodes.dataset.measurements ¦ WARNING ¦ measurements ¦ __exit__ ¦ 758 ¦ An exception occurred in measurement with guid: f6c56316-0000-0000-0000-019ece21ca08;
Traceback:
Traceback (most recent call last):
  File "D:\SNSPD\SNSPD2\snspd.py", line 483, in MSO5_counts_vs_current
    raise Exception('Error: Clipping')
Exception: Error: Clipping



Exception: Error: Clipping

Clipping error arose I think because the vertical scale was set low before the device had actually latched.

In [88]:
############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: False


In [90]:
idx_list

NameError: name 'idx_list' is not defined

In [94]:
params.device_line_2['currents'][39]

-9.75e-06

In [96]:
len(params.device_line_2['currents'])

53

In [100]:

############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')
time.sleep(10)

params.MSO5_counts_vs_current(device=params.device_line_2, n_captures=10, interval=1, osc=MS, dmm=dmm, yoko=yoko, pmeter90=pms120, unlatch=False, idx_list=np.arange(39,len(params.device_line_2['currents'])))

############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: True
Set standard oscilloscope parameters for counts
update station
Starting experimental run with id: 12. 
12


  0%|                                                                                           | 0/14 [00:00<?, ?it/s]

0.111
This acquisition will take 10s
12 26


2026-06-16 12:26:43,228 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()

  7%|█████▉                                                                             | 1/14 [00:22<04:49, 22.25s/it]

0.111
This acquisition will take 10s
12 26


...

0.111
This acquisition will take 10s
12 30


2026-06-16 12:31:09,460 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()

 93%|████████████████████████████████████████████████████████████████████████████▏     | 13/14 [04:48<00:22, 22.17s/it]

0.111
This acquisition will take 10s
12 31


2026-06-16 12:31:31,623 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()

100%|██████████████████████████████████████████████████████████████████████████████████| 14/14 [05:10<00:00, 22.18s/it]


Ramping to 0
Unlatch wait time 60
Laser enable status: False


Running from previous current where it ended. It appears to have rescaled to be half. 

Also 5V with both attenuators in line - counts are probably low because this is very small attenuation. 

In [62]:
MS.horizontal_scale()*MS.horizontal_divisions()

0.1

Disconnecting everything to see if it makes a difference. 2:36pm 